# Model Training

This notebook trains a few strong baseline classifiers and compares them using cross-validation.

All preprocessing is inside each model pipeline. That keeps training and prediction consistent and prevents the test set from influencing preprocessing.

## What are we predicting?

The target column is `status`. Its four classes are `operating`, `acquired`, `closed`, and `ipo`.

This is a multi-class classification problem because one company can belong to one of several outcome categories. The target encoder changes these text labels to integers for model training, then converts predictions back to the original status names.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier


def find_project_root() -> Path:
    current_folder = Path.cwd().resolve()
    for folder in [current_folder, *current_folder.parents]:
        if (folder / "src" / "data" / "companies.csv").exists():
            return folder
    raise FileNotFoundError("Could not find src/data/companies.csv")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "src" / "data" / "companies.csv"
TARGET_COLUMN = "status"
valid_statuses = ["operating", "acquired", "closed", "ipo"]
columns_to_drop = [
    TARGET_COLUMN, "id", "Unnamed: 0.1", "entity_type", "entity_id", "parent_id",
    "name", "normalized_name", "permalink", "domain", "homepage_url",
    "twitter_username", "logo_url", "logo_width", "logo_height", "short_description",
    "description", "overview", "tag_list", "created_by", "created_at", "updated_at",
    "first_investment_at", "last_investment_at", "first_funding_at", "last_funding_at",
    "first_milestone_at", "last_milestone_at", "closed_at", "ROI",
]

data = pd.read_csv(DATA_PATH)
data[TARGET_COLUMN] = data[TARGET_COLUMN].astype("string").str.strip().str.lower()
data = data[data[TARGET_COLUMN].isin(valid_statuses)].drop_duplicates().reset_index(drop=True)
X = data[[column for column in data.columns if column not in columns_to_drop]]

# XGBoost requires integer target labels, so keep a reversible encoder with the model.
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(data[TARGET_COLUMN])
class_names = target_encoder.classes_

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training rows: {len(X_train):,}; test rows: {len(X_test):,}")
print(f"Target classes: {list(class_names)}")

In [ ]:
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ],
)

models = {
    "logistic_regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
    ),
    "decision_tree": DecisionTreeClassifier(
        max_depth=12,
        min_samples_leaf=5,
        random_state=42,
        class_weight="balanced",
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=150,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced",
    ),
    "xgboost": XGBClassifier(
        n_estimators=150,
        max_depth=6,
        learning_rate=0.10,
        subsample=0.80,
        colsample_bytree=0.80,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1,
    ),
}

print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")
print(f"Models to compare: {', '.join(models)}")

In [ ]:
results = []
trained_pipelines = {}

for model_name, model in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=3,
        scoring={
            "accuracy": "accuracy",
            "f1_macro": "f1_macro",
            "f1_weighted": "f1_weighted",
        },
        n_jobs=-1,
    )
    results.append({
        "model": model_name,
        "mean_accuracy": scores["test_accuracy"].mean(),
        "mean_f1_macro": scores["test_f1_macro"].mean(),
        "mean_f1_weighted": scores["test_f1_weighted"].mean(),
    })
    trained_pipelines[model_name] = pipeline

comparison = pd.DataFrame(results).sort_values("mean_f1_macro", ascending=False)
comparison

In [ ]:
import pickle

best_model_name = comparison.iloc[0]["model"]
best_pipeline = trained_pipelines[best_model_name]
best_pipeline.fit(X_train, y_train)

model_path = PROJECT_ROOT / "src" / "models" / "best_model.pkl"
model_path.parent.mkdir(parents=True, exist_ok=True)
model_artifact = {
    "pipeline": best_pipeline,
    "target_encoder": target_encoder,
    "feature_columns": X.columns.tolist(),
    "model_name": best_model_name,
}
with model_path.open("wb") as model_file:
    pickle.dump(model_artifact, model_file)

print(f"Selected model: {best_model_name}")
print(f"Saved complete model artifact to: {model_path}")

## Training decision

The models are ranked by mean macro F1. Macro F1 gives every startup status equal importance, which is important because `operating` is much more common than the other classes.

Accuracy and weighted F1 are also reported, but they are not used as the only decision because a model could score well by mainly predicting the majority class.

Potential post-outcome fields such as `closed_at` and `ROI` are excluded before training to reduce target leakage.

The saved pickle contains the selected pipeline, the target-label encoder, and the feature names. This lets the application convert predictions back to readable statuses without a separate scaler or encoder file.